In [ ]:
import os
# Set NEAFE_DATA_FILE to the original station workbook before execution.


In [ ]:
import pandas as pd
import numpy as np
import featuretools as ft
import time
import warnings
warnings.filterwarnings('ignore')

print("开始 Featuretools 特征工程...")

# ==============================
# 1. 读取数据
# ==============================
data_path = os.environ["NEAFE_DATA_FILE"]
data = pd.read_excel(data_path, sheet_name="Sheet2")
data = data.dropna().reset_index(drop=True)

print("原始数据形状:", data.shape)

datetime_col = data.columns[0]
target_col = data.columns[-1]
data[datetime_col] = pd.to_datetime(data[datetime_col])

# ==============================
# 2. 时间序列拆分 (60/20/20)
# ==============================
split_idx_1 = int(len(data) * 0.6)
split_idx_2 = int(len(data) * 0.8)

train_df = data.iloc[:split_idx_1].copy()
valid_df = data.iloc[split_idx_1:split_idx_2].copy()
test_df = data.iloc[split_idx_2:].copy()

print(f"Train: {train_df.shape}, Valid: {valid_df.shape}, Test: {test_df.shape}")

# ==============================
# 3. Featuretools 特征工程（带 cutoff_time 防泄露）
# ==============================
def create_cutoff_times(df, datetime_col, instance_id_col='index'):
    """创建 cutoff_time 表（关键防泄露）"""
    cutoff_times = df[[datetime_col]].copy()
    cutoff_times = cutoff_times.reset_index()
    cutoff_times.columns = [instance_id_col, 'time']
    return cutoff_times

start_fe = time.time()

# --- 训练集生成特征定义 ---
train_df_ft = train_df.copy().reset_index()
train_cutoff_times = create_cutoff_times(train_df_ft, datetime_col)

es = ft.EntitySet(id="wind_data")
es = es.add_dataframe(
    dataframe_name="df",
    dataframe=train_df_ft,
    index="index",
    time_index=datetime_col,
    make_index=False
)

# 推荐原语（已优化）
trans_primitives = [
    "month", "weekday", "day", "hour", "minute",
    "is_weekend", "day_of_year",
    "cum_mean", "cum_sum", "cum_max", "cum_min"
]

agg_primitives = []   

train_ft, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="df",
    trans_primitives=trans_primitives,
    agg_primitives=agg_primitives,
    max_depth=1,
    cutoff_time=train_cutoff_times,
    cutoff_time_in_index=True,
    verbose=True
)

print(f"生成了 {len(feature_defs)} 个特征")

# --- 对验证集和测试集应用 ---
def apply_features(df, feature_defs, datetime_col):
    df_ft = df.copy().reset_index()
    cutoff_times = create_cutoff_times(df_ft, datetime_col)
    
    es_new = ft.EntitySet(id="new_data")
    es_new = es_new.add_dataframe(
        dataframe_name="df",
        dataframe=df_ft,
        index="index",
        time_index=datetime_col,
        make_index=False
    )
    
    fm = ft.calculate_feature_matrix(
        features=feature_defs,
        entityset=es_new,
        cutoff_time=cutoff_times,
        cutoff_time_in_index=True,
        verbose=False
    )
    return fm

valid_ft = apply_features(valid_df, feature_defs, datetime_col)
test_ft = apply_features(test_df, feature_defs, datetime_col)

end_fe = time.time()
print(f"Feature engineering 完成，用时: {end_fe - start_fe:.1f} 秒")

In [ ]:
# ==============================
# 4. 特征合并 + 清洗
# ==============================

# 保留原始数值特征
train_x = train_df.drop(columns=[target_col]).select_dtypes(include=[np.number]).reset_index(drop=True)
valid_x = valid_df.drop(columns=[target_col]).select_dtypes(include=[np.number]).reset_index(drop=True)
test_x = test_df.drop(columns=[target_col]).select_dtypes(include=[np.number]).reset_index(drop=True)

# FT 特征（去掉 target）
for df_ft in [train_ft, valid_ft, test_ft]:
    df_ft.drop(columns=[target_col], errors='ignore', inplace=True)

# 只保留数值特征（避免 category / object 报错）
train_ft = train_ft.select_dtypes(include=[np.number])
valid_ft = valid_ft.select_dtypes(include=[np.number])
test_ft  = test_ft.select_dtypes(include=[np.number])

# 合并 Raw + FT
train_x_new = pd.concat([train_x, train_ft.reset_index(drop=True)], axis=1)
valid_x_new = pd.concat([valid_x, valid_ft.reset_index(drop=True)], axis=1)
test_x_new  = pd.concat([test_x, test_ft.reset_index(drop=True)], axis=1)

# 删除重复列
train_x_new = train_x_new.loc[:, ~train_x_new.columns.duplicated()]
valid_x_new = valid_x_new.loc[:, ~valid_x_new.columns.duplicated()]
test_x_new  = test_x_new.loc[:, ~test_x_new.columns.duplicated()]

# ==============================
# 5. NaN / Inf 处理
# ==============================

for df in [train_x_new, valid_x_new, test_x_new]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 只用训练集统计量
train_mean = train_x_new.mean()

train_x_new = train_x_new.fillna(train_mean)
valid_x_new = valid_x_new.fillna(train_mean)
test_x_new  = test_x_new.fillna(train_mean)

# ==============================
# 6. 保存文件
# ==============================

train_x_new.reset_index(drop=True).to_feather("train_raw_FT_wind5.feather")
valid_x_new.reset_index(drop=True).to_feather("valid_raw_FT_wind5.feather")
test_x_new.reset_index(drop=True).to_feather("test_raw_FT_wind5.feather")

train_df[target_col].to_frame().reset_index(drop=True).to_feather("train_y_wind5.feather")
valid_df[target_col].to_frame().reset_index(drop=True).to_feather("valid_y_wind5.feather")
test_df[target_col].to_frame().reset_index(drop=True).to_feather("test_y_wind5.feather")

# 同时保存纯原始特征
train_x.to_feather("train_raw_wind5.feather")
valid_x.to_feather("valid_raw_wind5.feather")
test_x.to_feather("test_raw_wind5.feather")

print("所有特征工程和保存完成！")
print(f"最终特征数量: {train_x_new.shape[1]}")

In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Raw
train_raw = pd.read_feather("train_raw_wind5.feather")
valid_raw = pd.read_feather("valid_raw_wind5.feather")
test_raw  = pd.read_feather("test_raw_wind5.feather")

# NE-AFE
train_neafe = pd.read_feather("train_raw_FT_wind5.feather")
valid_neafe = pd.read_feather("valid_raw_FT_wind5.feather")
test_neafe  = pd.read_feather("test_raw_FT_wind5.feather")

# Label
train_y = pd.read_feather("train_y_wind5.feather").values.ravel()
valid_y = pd.read_feather("valid_y_wind5.feather").values.ravel()
test_y  = pd.read_feather("test_y_wind5.feather").values.ravel()

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    nrmse = rmse / (y_true.max() - y_true.min())
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "MSE": mse,
        "nRMSE": nrmse,
        "R2": r2
    }

In [ ]:
from xgboost import XGBRegressor

def run_xgb(train_x, valid_x, test_x, tag):

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "random_state": 42,
            "n_jobs": 4,
            "tree_method": "hist"  
        }

        model = XGBRegressor(**params)

        model.fit(
            train_x, train_y,
            eval_set=[(valid_x, valid_y)],
            verbose=False
        )

        pred = model.predict(valid_x)
        return np.sqrt(mean_squared_error(valid_y, pred))


    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10)

    best_params = study.best_params
    best_params.update({
        "random_state": 42,
        "n_jobs": 4,
        "tree_method": "hist"
    })

    # 重新训练
    best_model = XGBRegressor(**best_params)

    best_model.fit(
        train_x, train_y,
        eval_set=[(valid_x, valid_y)],
        verbose=False
    )

    preds = best_model.predict(test_x)
    metrics = evaluate(test_y, preds)
    
    print("Best Params:", best_params)
    print("Test Metrics:", metrics)
    
    return metrics 

In [ ]:
xgb_raw = run_xgb(train_raw, valid_raw, test_raw, "Raw")
xgb_ft = run_xgb(train_neafe, valid_neafe, test_neafe, "NE-AFE")
print("xgb Raw:", xgb_raw)
print("xgb FT:", xgb_ft)